# Expectations vs Realizations

Loads the extrapolated expectation series and plots them against actual
realized values from the Shiller dataset.

**The key question:** do survey expectations predict or correlate with
subsequent realized outcomes? This is both a validation of the extrapolated
series and a substantive finding about expectation formation.

**Four series:**
- Dividend growth expectations vs realized dividend growth (Shiller D)
- Earnings growth expectations vs realized earnings growth (Shiller E)
- 1-year return expectations vs realized 1-year S&P 500 returns (Shiller P)
- 10-year return expectations vs realized 10-year annualized returns

**Realized values** are computed from the Shiller monthly dataset aggregated
to quarterly frequency, aligned to the same wave dates as the expectations.

**Does not require main pipeline in memory** — loads everything from disk.

## 1. Configuration and paths

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ── Paths ─────────────────────────────────────────────────────────────────────
OUTPUT_DIR  = Path('./output')
DATA_DIR    = Path('./data')
SHILLER_XLS = DATA_DIR / 'ie_data.xls'

# ── Extrapolated series CSVs ──────────────────────────────────────────────────
SERIES_PATHS = {
    'dividend_growth': OUTPUT_DIR / 'dividend_growth' / 'extrapolated_dividend_growth.csv',
    'earnings_growth': OUTPUT_DIR / 'earnings_growth' / 'extrapolated_earnings_growth.csv',
    'returns_1yr':     OUTPUT_DIR / 'returns_1yr'     / 'extrapolated_returns_1yr.csv',
    'returns_10yr':    OUTPUT_DIR / 'returns_10yr'    / 'extrapolated_returns_10yr.csv',
}

# ── Horizon for realized returns (quarters ahead) ────────────────────────────
HORIZON_1YR  = 4    # 1-year = 4 quarters ahead
HORIZON_10YR = 40   # 10-year = 40 quarters ahead

# ── NBER recession dates ──────────────────────────────────────────────────────
NBER_RECESSIONS = [
    ('1926-10', '1927-11'), ('1929-08', '1933-03'), ('1937-05', '1938-06'),
    ('1945-02', '1945-10'), ('1948-11', '1949-10'), ('1953-07', '1954-05'),
    ('1957-08', '1958-04'), ('1960-04', '1961-02'), ('1969-12', '1970-11'),
    ('1973-11', '1975-03'), ('1980-01', '1980-07'), ('1981-07', '1982-11'),
    ('1990-07', '1991-03'), ('2001-03', '2001-11'), ('2007-12', '2009-06'),
    ('2020-02', '2020-04'),
]

print('Config loaded.')
for name, path in SERIES_PATHS.items():
    status = 'OK' if path.exists() else 'MISSING'
    print(f'  {name:<20} {status}  {path}')

## 2. Load Shiller data and compute realized series

In [ ]:
def load_shiller_realized(path=SHILLER_XLS):
    df = pd.read_excel(path, engine='xlrd', sheet_name='Data', header=7)
    df = df.dropna(subset=['Date'])
    df = df[df['Date'].astype(str).str.match(r'^\d{4}\.\d')].copy()

    def parse_date(d):
        d = float(d); year = int(d)
        month = max(1, min(12, int(round((d - year) * 100))))
        return pd.Timestamp(year=year, month=month, day=1)

    df['date'] = df['Date'].apply(parse_date)
    df = df.sort_values('date').reset_index(drop=True)

    df['P']   = pd.to_numeric(df['P'],         errors='coerce').ffill()
    df['D']   = pd.to_numeric(df['D'],         errors='coerce').ffill()
    df['E']   = pd.to_numeric(df['E'],         errors='coerce').ffill()
    df['CPI'] = pd.to_numeric(df['CPI'],       errors='coerce').ffill()

    # Quarter-end: take last month of each quarter
    df['quarter'] = df['date'].dt.to_period('Q')
    qdf = df.groupby('quarter').last().reset_index()
    qdf['wave_date'] = qdf['quarter'].dt.to_timestamp('Q')

    # ── Realized dividend growth (YoY, annualized) ───────────────────────────
    # Shiller D is 12-month trailing sum — pct_change(4) gives year-over-year
    qdf['div_gr_realized'] = qdf['D'].pct_change(4) * 100  # in percent

    # ── Realized earnings growth (YoY, annualized) ───────────────────────────
    qdf['earn_gr_realized'] = qdf['E'].pct_change(4) * 100

    # ── Realized 1-year total return ─────────────────────────────────────────
    # Total return = (P_{t+4} + D_{t+4}) / P_t - 1
    # Shiller D is already the 12-month trailing dividend sum — use it directly
    # as the income received over the forward year (approximation).
    # Multiply by 100 to convert to percent matching survey units.
    qdf['ret_1yr_realized'] = (
        (qdf['P'].shift(-4) + qdf['D'].shift(-4)) / qdf['P'] - 1
    ) * 100

    # ── Realized 10-year annualized real total return ─────────────────────────
    # Real price and dividend (CPI deflated)
    qdf['real_P'] = qdf['P'] / qdf['CPI']
    qdf['real_D'] = qdf['D'] / qdf['CPI']
    # Annualized: [(P_{t+40} + D_{t+40}) / P_t] ^ (1/10) - 1
    qdf['ret_10yr_realized'] = (
        ((qdf['real_P'].shift(-40) + qdf['real_D'].shift(-40))
         / qdf['real_P']) ** (1/10) - 1
    ) * 100

    print(f'Shiller quarterly data: {len(qdf)} quarters')
    print(f'  Date range: {qdf["wave_date"].min().date()} - {qdf["wave_date"].max().date()}')
    print(f'  1yr return range: {qdf["ret_1yr_realized"].min():.1f}% to {qdf["ret_1yr_realized"].max():.1f}%')
    print(f'  10yr return range: {qdf["ret_10yr_realized"].min():.1f}% to {qdf["ret_10yr_realized"].max():.1f}%')
    return qdf[['wave_date','div_gr_realized','earn_gr_realized',
                 'ret_1yr_realized','ret_10yr_realized']]

shiller_q = load_shiller_realized()
shiller_q.tail()

## 3. Load and merge expectation series with realized values

In [ ]:
def load_expectation_series(name, path, shiller_q):
    df = pd.read_csv(path, parse_dates=['wave_date'])
    df['wave_date'] = pd.to_datetime(df['wave_date'])

    # ── Cap earnings outlier: realized > 5 → cap at 2 ────────────────────────
    # One observed earnings value of ~8 compresses the y-axis of the chart.
    if name == 'earnings_growth' and 'realized' in df.columns:
        n_capped = (df['realized'] > 5).sum()
        if n_capped:
            print(f'  Capping {n_capped} earnings outlier(s) >5 to 2')
            df.loc[df['realized'] > 5, 'realized'] = 2.0

    realized_col = {
        'dividend_growth': 'div_gr_realized',
        'earnings_growth': 'earn_gr_realized',
        'returns_1yr':     'ret_1yr_realized',
        'returns_10yr':    'ret_10yr_realized',
    }[name]

    # Merge Shiller realized values
    df = df.merge(
        shiller_q[['wave_date', realized_col]],
        on='wave_date', how='left'
    ).rename(columns={realized_col: 'realized_actual'})

    print(f'{name}: {len(df)} waves  '
          f'({df["wave_date"].min().date()} - {df["wave_date"].max().date()})  '
          f'realized_actual coverage: {df["realized_actual"].notna().sum()}  '
          f'survey realized coverage: {df["realized"].notna().sum() if "realized" in df.columns else 0}')
    return df


loaded = {}
for name, path in SERIES_PATHS.items():
    if not path.exists():
        print(f'SKIP {name}: file not found')
        continue
    loaded[name] = load_expectation_series(name, path, shiller_q)

## 4. Main comparison plot

For each series: predicted expectations (blue) overlaid on realized outcomes
(red). Recession shading in gray. Survey sample start marked with dashed line.

**Interpretation note:** For returns, realized values are shown with the
appropriate lead — 1-year returns are the actual return earned over the
following year, 10-year returns are the annualized real return over the
following decade. A good model should show expectations leading realizations.

In [ ]:
SERIES_META = {
    'dividend_growth': {
        'title':     'Dividend Growth',
        'ylabel':    'Annual dividend growth (%)',
        'exp_label': 'Expected (extrapolated)',
        'rea_label': 'Realized (Shiller)',
        'sur_label': 'Realized (survey)',
        'scale':     1.0,
    },
    'earnings_growth': {
        'title':     'Earnings Growth',
        'ylabel':    'Annual earnings growth (%)',
        'exp_label': 'Expected (extrapolated)',
        'rea_label': 'Realized (Shiller)',
        'sur_label': 'Realized (survey)',
        'scale':     1.0,
    },
    'returns_1yr': {
        'title':     '1-Year S&P 500 Return',
        'ylabel':    'Return (%)',
        'exp_label': 'Expected (extrapolated)',
        'rea_label': 'Realized 1-yr total return (Shiller)',
        'sur_label': 'Realized (survey)',
        'scale':     1.0,   # both predicted and realized already in percent
    },
    'returns_10yr': {
        'title':     '10-Year S&P 500 Return (Annualized Real)',
        'ylabel':    'Annualized real return (%)',
        'exp_label': 'Expected (extrapolated)',
        'rea_label': 'Realized 10-yr ann. real return (Shiller)',
        'sur_label': 'Realized (survey)',
        'scale':     1.0,
    },
}

def shade_recessions(ax, xmin, xmax):
    for start, end in NBER_RECESSIONS:
        s = pd.Timestamp(start + '-01')
        e = pd.Timestamp(end   + '-01')
        if e >= xmin and s <= xmax:
            ax.axvspan(s, e, alpha=0.12, color='gray', zorder=0)


n_series = len(loaded)
fig, axes = plt.subplots(n_series, 1, figsize=(16, 5.5 * n_series))
if n_series == 1: axes = [axes]
fig.suptitle('Survey Expectations vs Realized Outcomes', fontweight='bold', fontsize=14)

for ax, (name, df) in zip(axes, loaded.items()):
    meta  = SERIES_META.get(name, {})
    scale = meta.get('scale', 1.0)

    xmin = df['wave_date'].min()
    xmax = df['wave_date'].max()
    shade_recessions(ax, xmin, xmax)

    # OOS shading
    if 'out_of_support' in df.columns:
        for _, row in df[df['out_of_support']].iterrows():
            ax.axvspan(row['wave_date'] - pd.Timedelta(days=45),
                       row['wave_date'] + pd.Timedelta(days=45),
                       alpha=0.10, color='#e07b39', zorder=0)

    # ── Realized (Shiller) — red ──────────────────────────────────────────────
    real = df.dropna(subset=['realized_actual'])
    ax.plot(real['wave_date'], real['realized_actual'],
            '-', color='#c0392b', lw=1.2, alpha=0.75,
            label=meta.get('rea_label','Realized (Shiller)'), zorder=4)

    # ── Survey realized — green (calibration period only) ────────────────────
    if 'realized' in df.columns:
        surv = df[df['in_calibration']].dropna(subset=['realized'])
        if len(surv):
            ax.plot(surv['wave_date'], surv['realized'] * scale,
                    'o-', color='#27ae60', lw=1.5, ms=3, alpha=0.9,
                    label=meta.get('sur_label','Realized (survey)'), zorder=6)

    # ── Predicted expectations — blue ─────────────────────────────────────────
    pre = df[~df['in_calibration']]
    cal = df[ df['in_calibration']]
    ax.plot(pre['wave_date'], pre['predicted'] * scale,
            '-', color='#2c5f8a', lw=1.5, alpha=0.6,
            label=meta.get('exp_label','Expected') + ' (pre-survey)', zorder=5)
    ax.plot(cal['wave_date'], cal['predicted'] * scale,
            '-', color='#2c5f8a', lw=2.0, alpha=0.95,
            label=meta.get('exp_label','Expected') + ' (calibration)', zorder=5)

    # Survey start line
    if len(cal):
        ax.axvline(cal['wave_date'].min(), color='black', ls='--',
                   lw=1, alpha=0.5, label='Survey sample start')

    ax.set_title(meta.get('title', name), fontweight='bold', fontsize=11)
    ax.set_ylabel(meta.get('ylabel', ''))
    ax.grid(alpha=0.25)

    handles, labels = ax.get_legend_handles_labels()
    handles.append(mpatches.Patch(color='gray',    alpha=0.3, label='NBER recession'))
    handles.append(mpatches.Patch(color='#e07b39', alpha=0.3, label='Out of support'))
    ax.legend(handles=handles, fontsize=7.5, loc='upper right', ncol=2)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'expectations_vs_realized.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved expectations_vs_realized.png')

## 5. Correlation and predictive power analysis

For each series, compute the correlation between expectations and
contemporaneous realized values (co-movement) and between expectations
and *subsequent* realized values (predictive content).

A high contemporaneous correlation validates that the expectations series
tracks real-world conditions. A high predictive correlation would suggest
the survey expectations contain forward-looking information.

In [ ]:
print('\n── Correlation Analysis ──────────────────────────────────────────────────')
print(f'{"Series":<22} {"Contemp r":>10} {"Contemp p":>10} '
      f'{"Lead-1Q r":>10} {"Lead-4Q r":>10} {"N":>6}')
print('─' * 75)

corr_results = {}

for name, df in loaded.items():
    # Align on waves that have both expectation and realized
    clean = df.dropna(subset=['predicted', 'realized_actual']).copy()
    if len(clean) < 10:
        print(f'{name:<22} insufficient overlap')
        continue

    scale = SERIES_META.get(name, {}).get('scale', 1.0)
    exp   = clean['predicted'].values * scale
    rea   = clean['realized_actual'].values

    # Contemporaneous correlation
    r_contemp, p_contemp = stats.pearsonr(exp, rea)

    # Predictive: expectations at t vs realized at t+1 and t+4
    # Merge df with itself shifted
    df2 = df[['wave_date','predicted','realized_actual']].copy()
    df2['exp_scaled'] = df2['predicted'] * scale
    df2 = df2.sort_values('wave_date').reset_index(drop=True)

    # Lead 1 quarter
    df2['realized_lead1'] = df2['realized_actual'].shift(-1)
    lead1 = df2.dropna(subset=['exp_scaled','realized_lead1'])
    r_lead1 = stats.pearsonr(lead1['exp_scaled'], lead1['realized_lead1'])[0] \
              if len(lead1) > 5 else np.nan

    # Lead 4 quarters
    df2['realized_lead4'] = df2['realized_actual'].shift(-4)
    lead4 = df2.dropna(subset=['exp_scaled','realized_lead4'])
    r_lead4 = stats.pearsonr(lead4['exp_scaled'], lead4['realized_lead4'])[0] \
              if len(lead4) > 5 else np.nan

    corr_results[name] = {
        'r_contemp': r_contemp, 'p_contemp': p_contemp,
        'r_lead1': r_lead1, 'r_lead4': r_lead4, 'n': len(clean),
    }

    sig = '**' if p_contemp < 0.01 else '*' if p_contemp < 0.05 else ''
    print(f'{name:<22} {r_contemp:>9.3f}{sig} {p_contemp:>10.4f} '
          f'{r_lead1:>10.3f} {r_lead4:>10.3f} {len(clean):>6}')

print('\n* p<0.05  ** p<0.01')
print('Lead-1Q: correlation of expectation at t with realized at t+1')
print('Lead-4Q: correlation of expectation at t with realized at t+4')

In [ ]:
# ── Scatter plots: expectation vs realized ────────────────────────────────────
n_series = len(loaded)
n_cols   = min(2, n_series)
n_rows   = (n_series + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols,
                          figsize=(7 * n_cols, 6 * n_rows), squeeze=False)
fig.suptitle('Expectations vs Realized — Scatter', fontweight='bold', fontsize=13)

for idx, (name, df) in enumerate(loaded.items()):
    ax    = axes[idx // n_cols, idx % n_cols]
    meta  = SERIES_META.get(name, {})
    scale = meta.get('scale', 1.0)
    clean = df.dropna(subset=['predicted','realized_actual'])
    if len(clean) == 0:
        ax.set_visible(False); continue

    exp = clean['predicted'].values * scale
    rea = clean['realized_actual'].values

    # Color by in_calibration
    colors = ['#2c5f8a' if ic else '#aec7e8'
              for ic in clean['in_calibration']]

    ax.scatter(exp, rea, c=colors, alpha=0.65, s=20, zorder=3)
    # OLS line
    m, b, r, p, se = stats.linregress(exp, rea)
    xr = np.linspace(exp.min(), exp.max(), 100)
    ax.plot(xr, m*xr+b, '--', color='#c0392b', lw=1.5,
            label=f'OLS  r={r:.3f}  p={p:.3f}')
    ax.axline((0,0), slope=1, color='gray', lw=0.8, ls=':', alpha=0.5,
              label='45° line')

    ax.set_xlabel(meta.get('exp_label','Expected'))
    ax.set_ylabel(meta.get('rea_label','Realized'))
    ax.set_title(meta.get('title', name), fontweight='bold')
    ax.legend(fontsize=8); ax.grid(alpha=0.25)

    # Add legend patches
    from matplotlib.lines import Line2D
    handles, labels = ax.get_legend_handles_labels()
    handles += [Line2D([0],[0], marker='o', color='w', markerfacecolor='#2c5f8a',
                       ms=7, label='Calibration period'),
                Line2D([0],[0], marker='o', color='w', markerfacecolor='#aec7e8',
                       ms=7, label='Pre-survey')]
    ax.legend(handles=handles, fontsize=7.5)

for idx in range(len(loaded), n_rows * n_cols):
    axes[idx // n_cols, idx % n_cols].set_visible(False)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'expectations_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Rolling correlation over time

How stable is the expectations-realizations relationship across different
macro regimes? A 20-quarter rolling window shows whether the correlation
varies with economic conditions.

In [ ]:
ROLL = 20  # quarters

n_series = len(loaded)
n_cols   = min(2, n_series)
n_rows   = (n_series + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols,
                          figsize=(10 * n_cols, 4.5 * n_rows), squeeze=False)
fig.suptitle(f'Rolling {ROLL}-quarter correlation: Expectations vs Realized',
             fontweight='bold', fontsize=12)

for idx, (name, df) in enumerate(loaded.items()):
    ax    = axes[idx // n_cols, idx % n_cols]
    meta  = SERIES_META.get(name, {})
    scale = meta.get('scale', 1.0)

    clean = df.dropna(subset=['predicted','realized_actual']).copy()
    clean = clean.sort_values('wave_date').reset_index(drop=True)
    clean['exp_scaled'] = clean['predicted'] * scale

    roll_corr = clean[['exp_scaled','realized_actual']].rolling(ROLL).corr()
    roll_corr = roll_corr.unstack()['exp_scaled']['realized_actual']

    shade_recessions(ax, clean['wave_date'].min(), clean['wave_date'].max())
    ax.plot(clean['wave_date'], roll_corr,
            color='#2c5f8a', lw=1.8, label=f'{ROLL}-quarter rolling r')
    ax.axhline(0, color='black', lw=0.8, ls='--')
    ax.axhline(0.3, color='gray', lw=0.8, ls=':', alpha=0.6)
    ax.axhline(-0.3, color='gray', lw=0.8, ls=':', alpha=0.6)

    if 'in_calibration' in clean.columns:
        first_cal = clean[clean['in_calibration']]['wave_date'].min()
        if pd.notna(first_cal):
            ax.axvline(first_cal, color='black', ls='--', lw=1, alpha=0.5,
                       label='Survey start')

    ax.set_title(meta.get('title', name), fontweight='bold')
    ax.set_ylabel('Pearson r'); ax.set_xlabel('Wave date')
    ax.set_ylim(-1, 1)
    ax.legend(fontsize=8); ax.grid(alpha=0.25)

for idx in range(len(loaded), n_rows * n_cols):
    axes[idx // n_cols, idx % n_cols].set_visible(False)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'rolling_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Final summary table ───────────────────────────────────────────────────────
print('\n── Summary ───────────────────────────────────────────────────────────────')
print(f'{"Series":<22} {"N":>6} {"Contemp r":>10} {"Lead-1Q r":>11} {"Lead-4Q r":>11}')
print('─' * 65)
for name, cr in corr_results.items():
    print(f'{name:<22} {cr["n"]:>6} '
          f'{cr["r_contemp"]:>10.3f} '
          f'{cr["r_lead1"]:>11.3f} '
          f'{cr["r_lead4"]:>11.3f}')
print()
print('Contemp r : Pearson correlation between expectation and contemporaneous realized')
print('Lead-1Q r : expectation at t vs realized at t+1 (1-quarter ahead)')
print('Lead-4Q r : expectation at t vs realized at t+4 (1-year ahead)')
print()
print('High contemp r  → expectation series co-moves with fundamentals (validation)')
print('High lead r     → expectations contain forward-looking information')